<a href="https://colab.research.google.com/github/jeremy26/hydranets_course/blob/claude/modernize-autoware-course-edY16/Module_2_Depth_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: HydraNet — Depth + Segmentation

In this module, you'll build a **modern HydraNet** that performs **semantic segmentation** and **monocular depth estimation** simultaneously from a single camera image.

**Architecture** (based on [Autoware Vision Pilot](https://github.com/autowarefoundation/autoware_vision_pilot)):

```
Image -> EfficientNet-B0 Backbone -> Context Module -> Shared Neck -> Seg Head
                                                                   -> Depth Head
```

**Key concepts:**
- Shared backbone with multi-scale features
- Context modules for global scene understanding (pseudo-attention)
- U-Net style decoder neck with skip connections
- Task-specific lightweight heads
- Multi-task loss balancing

**Dataset:** BDD100K (driving scenes) + Depth Anything v2 pseudo-depth labels

# 1 — Setup & Installation

In [ ]:
# Clone the course repo and install dependencies
!git clone -b claude/modernize-autoware-course-edY16 https://github.com/jeremy26/hydranets_course.git 2>/dev/null || true
%cd hydranets_course

!pip install -q torchvision pillow matplotlib numpy tqdm

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2 — Dataset: BDD100K + Pseudo-Depth

We use [BDD100K](https://www.vis.xyz/bdd100k/), a large-scale driving dataset with:
- **10K images** with pixel-level semantic segmentation (19 classes, Cityscapes-compatible)
- **100K images** with bounding boxes, lane markings, GPS/IMU

BDD100K doesn't include depth annotations, so we generate **pseudo-depth labels** using [Depth Anything v2](https://github.com/DepthAnything/Depth-Anything-V2) — a state-of-the-art monocular depth estimation foundation model. This is a common practice called **knowledge distillation**.

**Two ways to get the data:**
- **Kaggle (recommended):** Add the datasets to your notebook — zero download time
- **Colab/local:** `wget` from the [ETH Zurich mirror](https://dl.cv.ethz.ch/bdd100k/data/)

In [ ]:
import os

# ============================================================
# OPTION 1: Kaggle (recommended — instant, no download)
# Add these datasets to your Kaggle notebook:
#   - solesensei/solesensei_bdd100k  (images + all label types)
#   - thakurmayank5/depth-images-for-bdd100k-dataset  (depth maps)
# ============================================================
KAGGLE_PATHS = [
    "/kaggle/input/solesensei-bdd100k",       # solesensei (hyphens in mount)
    "/kaggle/input/solesensei_bdd100k",        # solesensei (underscores)
    "/kaggle/input/bdd100k-dataset",           # awsaf49
]

DEPTH_PATHS = [
    "/kaggle/input/depth-images-for-bdd100k-dataset",
]

DATA_ROOT = None
DEPTH_ROOT = None

for p in KAGGLE_PATHS:
    if os.path.exists(p):
        DATA_ROOT = p
        break

for p in DEPTH_PATHS:
    if os.path.exists(p):
        DEPTH_ROOT = p
        break

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f"Running on Kaggle — BDD100K mounted at {DATA_ROOT}")
    if DEPTH_ROOT:
        print(f"  Depth dataset mounted at {DEPTH_ROOT}")

# ============================================================
# OPTION 2: Colab / Local — download via Kaggle API
# Requires: kaggle.json credentials (or Colab Kaggle secret)
# ============================================================
else:
    DATA_ROOT = "data/bdd100k"
    DEPTH_ROOT = os.path.join(DATA_ROOT, "labels", "depth")
    os.makedirs(DATA_ROOT, exist_ok=True)

    # Install kaggle CLI if not present
    !pip install -q kaggle

    # 1. BDD100K images + labels (solesensei mirror)
    print("Downloading BDD100K from Kaggle (solesensei/solesensei_bdd100k)...")
    !kaggle datasets download -d solesensei/solesensei_bdd100k -p data/ --unzip

    # 2. Pseudo-depth labels (Depth Anything v2)
    print("\nDownloading depth maps from Kaggle...")
    !kaggle datasets download -d thakurmayank5/depth-images-for-bdd100k-dataset -p {DEPTH_ROOT} --unzip

    print("\nDownloads complete!")

# Verify dataset structure
print(f"\nDATA_ROOT: {DATA_ROOT}")
for subdir in ['images/10k/train', 'images/10k/val',
               'labels/sem_seg/masks/train', 'labels/sem_seg/masks/val']:
    path = os.path.join(DATA_ROOT, subdir)
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"  {subdir}: {count} files")
    else:
        print(f"  {subdir}: NOT FOUND")


## Visualize the Data

Let's look at a few examples: RGB image, segmentation mask, and pseudo-depth.

In [ ]:
# BDD100K / Cityscapes color palette (19 classes)
BDD_COLORS = np.array([
    [128, 64,128],  # road
    [244, 35,232],  # sidewalk
    [ 70, 70, 70],  # building
    [102,102,156],  # wall
    [190,153,153],  # fence
    [153,153,153],  # pole
    [250,170, 30],  # traffic light
    [220,220,  0],  # traffic sign
    [107,142, 35],  # vegetation
    [152,251,152],  # terrain
    [ 70,130,180],  # sky
    [220, 20, 60],  # person
    [255,  0,  0],  # rider
    [  0,  0,142],  # car
    [  0,  0, 70],  # truck
    [  0, 60,100],  # bus
    [  0, 80,100],  # train
    [  0,  0,230],  # motorcycle
    [119, 11, 32],  # bicycle
], dtype=np.uint8)

BDD_CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
               'traffic light', 'traffic sign', 'vegetation', 'terrain',
               'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
               'motorcycle', 'bicycle']

def colorize_mask(mask, colors=BDD_COLORS):
    """Convert a class-index mask to an RGB image."""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(len(colors)):
        rgb[mask == cls_id] = colors[cls_id]
    return rgb

In [ ]:
# Visualize a few samples
img_dir = os.path.join(DATA_ROOT, 'images', '10k', 'train')
if not os.path.exists(img_dir):
    img_dir = os.path.join(DATA_ROOT, 'images', 'train')

seg_dir = os.path.join(DATA_ROOT, 'labels', 'sem_seg', 'masks', 'train')
depth_dir = os.path.join(DATA_ROOT, 'labels', 'depth', 'train')

images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for i in range(3):
    idx = np.random.randint(0, len(images))
    basename = os.path.splitext(images[idx])[0]

    # RGB
    img = Image.open(os.path.join(img_dir, images[idx]))
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('RGB Image')
    axes[i, 0].axis('off')

    # Segmentation
    seg_path = os.path.join(seg_dir, basename + '.png')
    if os.path.exists(seg_path):
        seg = np.array(Image.open(seg_path))
        axes[i, 1].imshow(colorize_mask(seg))
    axes[i, 1].set_title('Segmentation')
    axes[i, 1].axis('off')

    # Depth
    depth_path = os.path.join(depth_dir, basename + '.npy')
    if os.path.exists(depth_path):
        depth = np.load(depth_path)
        axes[i, 2].imshow(depth, cmap='magma')
    axes[i, 2].set_title('Pseudo-Depth (Depth Anything v2)')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## DataLoader

We use our custom `BDD100KDataset` class that loads images, segmentation masks, and depth maps. All images are resized to **320x640** (a good balance between quality and Colab speed).

In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


# ImageNet normalization (used by EfficientNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Input resolution (H, W) — 320x640 is a good balance for Colab
INPUT_SIZE = (320, 640)


class BDD100KDataset(Dataset):
    """BDD100K dataset for multi-task learning.

    Supports loading any combination of:
    - Semantic segmentation masks
    - Pseudo-depth maps (from Depth Anything v2)
    - Lane segmentation masks

    Args:
        root_dir: path to BDD100K data root
        split: 'train' or 'val'
        tasks: list of tasks to load, e.g. ['seg', 'depth', 'lanes']
        depth_root: separate root for depth maps (if from a different dataset)
        common_images: optional list of image basenames to include (filters to only these)
        transform: optional additional transforms
    """

    def __init__(self, root_dir, split='train', tasks=None, depth_root=None,
                 common_images=None, transform=None):
        self.root_dir = root_dir
        self.split = split
        self.tasks = tasks or ['seg', 'depth']
        self.transform = transform

        # Image directory
        self.img_dir = os.path.join(root_dir, 'images', '10k', split)
        if not os.path.exists(self.img_dir):
            self.img_dir = os.path.join(root_dir, 'images', split)

        # Collect image filenames
        all_filenames = sorted([
            f for f in os.listdir(self.img_dir)
            if f.endswith(('.jpg', '.png'))
        ])

        # Filter to common images if provided
        if common_images is not None:
            common_set = set(common_images)
            self.filenames = [f for f in all_filenames if os.path.splitext(f)[0] in common_set or f in common_set]
        else:
            self.filenames = all_filenames

        # Segmentation mask directory
        self.seg_dir = os.path.join(root_dir, 'labels', 'sem_seg', 'masks', split)

        # Pseudo-depth directory (from separate Depth Anything v2 dataset or local)
        if depth_root:
            # Depth from a separate dataset — check for split subdirs or flat layout
            split_path = os.path.join(depth_root, split)
            self.depth_dir = split_path if os.path.exists(split_path) else depth_root
        else:
            self.depth_dir = os.path.join(root_dir, 'labels', 'depth', split)

        # Lane segmentation directory
        self.lane_dir = os.path.join(root_dir, 'labels', 'lane', 'masks', split)

        # Standard image transforms
        self.img_transform = transforms.Compose([
            transforms.Resize(INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        basename = os.path.splitext(filename)[0]

        # Load and transform image
        img_path = os.path.join(self.img_dir, filename)
        image = Image.open(img_path).convert('RGB')
        image = self.img_transform(image)

        sample = {'image': image, 'filename': basename}

        # Load segmentation mask
        if 'seg' in self.tasks:
            seg_path = os.path.join(self.seg_dir, basename + '.png')
            if os.path.exists(seg_path):
                seg = Image.open(seg_path)
                seg = seg.resize((INPUT_SIZE[1], INPUT_SIZE[0]),
                                 Image.NEAREST)
                seg = torch.from_numpy(np.array(seg)).long()
            else:
                seg = torch.zeros(INPUT_SIZE, dtype=torch.long)
            sample['seg'] = seg

        # Load depth map
        if 'depth' in self.tasks:
            depth_path = os.path.join(self.depth_dir, basename + '.npy')
            if os.path.exists(depth_path):
                depth = np.load(depth_path).astype(np.float32)
                depth = torch.from_numpy(depth).unsqueeze(0)
                depth = torch.nn.functional.interpolate(
                    depth.unsqueeze(0), size=INPUT_SIZE, mode='bilinear',
                    align_corners=False
                ).squeeze(0)
            else:
                depth = torch.zeros((1, *INPUT_SIZE), dtype=torch.float32)
            sample['depth'] = depth

        # Load lane mask
        if 'lanes' in self.tasks:
            lane_path = os.path.join(self.lane_dir, basename + '.png')
            if os.path.exists(lane_path):
                lane = Image.open(lane_path)
                lane = lane.resize(
                    (INPUT_SIZE[1] // 4, INPUT_SIZE[0] // 4),
                    Image.NEAREST
                )
                lane = torch.from_numpy(np.array(lane)).long()
            else:
                lane = torch.zeros(
                    (INPUT_SIZE[0] // 4, INPUT_SIZE[1] // 4),
                    dtype=torch.long
                )
            sample['lanes'] = lane

        return sample


def get_dataloaders(root_dir, tasks=None, batch_size=8, num_workers=2,
                    depth_root=None, common_images=None):
    """Create train and val dataloaders for BDD100K."""
    train_dataset = BDD100KDataset(
        root_dir, split='train', tasks=tasks,
    )
    val_dataset = BDD100KDataset(
        root_dir, split='val', tasks=tasks,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader


train_loader, val_loader = get_dataloaders(
    root_dir=DATA_ROOT,
    tasks=['seg', 'depth'],
    batch_size=8,
    num_workers=2,
    depth_root=DEPTH_ROOT,
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Input resolution: {INPUT_SIZE}")

# Check a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Image:  {batch['image'].shape}")
print(f"  Seg:    {batch['seg'].shape}")
print(f"  Depth:  {batch['depth'].shape}")


# 3 — Architecture: Building the HydraNet

Our HydraNet follows the **Autoware Vision Pilot** pattern. Every task goes through 4 stages:

```
Image ─> [Backbone] ─> [Context] ─> [Neck] ─> [Head] ─> Prediction
              │                        ▲
              └── skip connections ─────┘
```

| Component | Role | Trainable? |
|-----------|------|-----------|
| **Backbone** | EfficientNet-B0 — extracts multi-scale features | Pretrained, fine-tuned |
| **Context** | Global Average Pool → MLP → spatial reconstruction → pseudo-attention | Yes |
| **Neck** | U-Net decoder with skip connections from backbone | Yes (shared) |
| **Head** | Lightweight task-specific output layers | Yes (per-task) |

Let's build each component step by step.

## 3.1 — Backbone: EfficientNet-B0

We use **EfficientNet-B0** (pretrained on ImageNet) as our backbone encoder. Unlike the old MobileNetv2, EfficientNet uses compound scaling and is more efficient.

The backbone extracts features at **5 different scales** — these multi-scale features are crucial for the skip connections in the decoder.

In [ ]:
from torchvision import models


class Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(Backbone, self).__init__()
        if pretrained:
            self.encoder = models.efficientnet_b0(
                weights='EfficientNet_B0_Weights.IMAGENET1K_V1'
            ).features
        else:
            self.encoder = models.efficientnet_b0(weights=None).features

    def forward(self, image):
        # EfficientNet-B0 feature stages
        # Input: (B, 3, H, W)
        l0 = self.encoder[0](image)   # (B, 32, H/2, W/2)    - stride 2
        l1 = self.encoder[1](l0)      # (B, 16, H/2, W/2)    - stride 2
        l2 = self.encoder[2](l1)      # (B, 24, H/4, W/4)    - stride 4
        l3 = self.encoder[3](l2)      # (B, 40, H/8, W/8)    - stride 8
        l4 = self.encoder[4](l3)      # (B, 80, H/16, W/16)  - stride 16
        l5 = self.encoder[5](l4)      # (B, 112, H/16, W/16) - stride 16
        l6 = self.encoder[6](l5)      # (B, 192, H/32, W/32) - stride 32
        l7 = self.encoder[7](l6)      # (B, 320, H/32, W/32) - stride 32
        l8 = self.encoder[8](l7)      # (B, 1280, H/32, W/32)- stride 32

        # Return multi-scale features for skip connections
        # features[0] = low-level (high res), features[4] = deep (low res)
        return [l0, l2, l3, l4, l8]


backbone = Backbone(pretrained=True)

# Let's trace the feature dimensions through the backbone
dummy_input = torch.randn(1, 3, 320, 640)
features = backbone(dummy_input)

print("Backbone multi-scale features:")
print(f"{'Level':<10} {'Shape':<30} {'Use'}")
print("-" * 65)
for i, f in enumerate(features):
    use = ['skip to head (H/2)', 'skip to neck block 3 (H/4)',
           'skip to neck block 2 (H/8)', 'skip to neck block 1 (H/16)',
           'deep features -> context (H/32)'][i]
    print(f"feat[{i}]    {str(list(f.shape)):<30} {use}")

## 3.2 — Context Module: Global Scene Understanding

The Context module is what makes this architecture special. Before decoding, we ask: **"What kind of scene am I looking at?"**

It works in 3 steps:
1. **Global Average Pooling** — compress the entire feature map to a single vector
2. **MLP** — process through fully connected layers to capture scene-level patterns
3. **Spatial Reconstruction** — reshape back to a spatial feature map
4. **Pseudo-Attention** — `context * features + features` (like a residual gate)

This helps the network understand global context (e.g., "this is a highway" vs "this is an intersection") before making per-pixel predictions.

In [ ]:
class SceneContext(nn.Module):
    """Context module for scene segmentation.
    Processes 1280-channel deep features from EfficientNet-B0."""

    def __init__(self):
        super(SceneContext, self).__init__()
        self.GeLU = nn.GELU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.25)

        # MLP: compress deep features to a compact representation
        self.context_layer_0 = nn.Linear(1280, 800)
        self.context_layer_1 = nn.Linear(800, 800)
        self.context_layer_2 = nn.Linear(800, 200)

        # Spatial reconstruction: expand back to feature map
        self.context_layer_3 = nn.Conv2d(1, 128, 3, 1, 1)
        self.context_layer_4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.context_layer_5 = nn.Conv2d(256, 512, 3, 1, 1)
        self.context_layer_6 = nn.Conv2d(512, 1280, 3, 1, 1)

    def forward(self, features):
        # Global Average Pooling: (B, 1280, H, W) -> (B, 1280)
        feature_vector = torch.mean(features, dim=[2, 3])

        # MLP to capture scene-level context
        c0 = self.GeLU(self.dropout(self.context_layer_0(feature_vector)))
        c1 = self.GeLU(self.dropout(self.context_layer_1(c0)))
        c2 = self.sigmoid(self.dropout(self.context_layer_2(c1)))

        # Reshape to spatial: (B, 200) -> (B, 1, 10, 20)
        b = c2.shape[0]
        c3 = c2.view(b, 1, 10, 20)

        # Reconstruct to match deep feature dimensions
        c4 = self.GeLU(self.context_layer_3(c3))
        c5 = self.GeLU(self.context_layer_4(c4))
        c6 = self.GeLU(self.context_layer_5(c5))
        c7 = self.GeLU(self.context_layer_6(c6))

        # Pseudo-attention: multiply + residual
        context = c7 * features + features
        return context


class DepthContext(nn.Module):
    """Context module for depth estimation.
    Same structure as SceneContext — depth and segmentation
    benefit from different learned context representations."""

    def __init__(self):
        super(DepthContext, self).__init__()
        self.GeLU = nn.GELU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.25)

        self.context_layer_0 = nn.Linear(1280, 800)
        self.context_layer_1 = nn.Linear(800, 800)
        self.context_layer_2 = nn.Linear(800, 200)

        self.context_layer_3 = nn.Conv2d(1, 128, 3, 1, 1)
        self.context_layer_4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.context_layer_5 = nn.Conv2d(256, 512, 3, 1, 1)
        self.context_layer_6 = nn.Conv2d(512, 1280, 3, 1, 1)

    def forward(self, features):
        feature_vector = torch.mean(features, dim=[2, 3])

        c0 = self.GeLU(self.dropout(self.context_layer_0(feature_vector)))
        c1 = self.GeLU(self.dropout(self.context_layer_1(c0)))
        c2 = self.sigmoid(self.dropout(self.context_layer_2(c1)))

        b = c2.shape[0]
        c3 = c2.view(b, 1, 10, 20)

        c4 = self.GeLU(self.context_layer_3(c3))
        c5 = self.GeLU(self.context_layer_4(c4))
        c6 = self.GeLU(self.context_layer_5(c5))
        c7 = self.GeLU(self.context_layer_6(c6))

        context = c7 * features + features
        return context


seg_context = SceneContext()
depth_context = DepthContext()

# The context module takes the deepest features and returns attention-weighted features
deep_features = features[4]  # (1, 1280, 10, 20)
print(f"Input to context:  {list(deep_features.shape)}")

seg_ctx = seg_context(deep_features)
depth_ctx = depth_context(deep_features)
print(f"Seg context output:   {list(seg_ctx.shape)}")
print(f"Depth context output: {list(depth_ctx.shape)}")
print(f"\nNotice: same shape as input! The context module acts as an attention gate.")

## 3.3 — Neck: U-Net Style Decoder

The Neck upsamples the context-enhanced features back to higher resolution, using **skip connections** from the encoder at each stage.

```
context (1280ch, H/32) ──┐
                          ▼
               [ConvTranspose2d 2x]  +  features[3] (80ch, H/16)  ──> 768ch, H/16
                          ▼
               [ConvTranspose2d 2x]  +  features[2] (40ch, H/8)   ──> 512ch, H/8
                          ▼
               [ConvTranspose2d 2x]  +  features[1] (24ch, H/4)   ──> 256ch, H/4 = NECK OUTPUT
```

The neck is **shared** across tasks — both segmentation and depth use the same decoded features.

In [ ]:
class SceneNeck(nn.Module):
    def __init__(self):
        super(SceneNeck, self).__init__()
        self.GeLU = nn.GELU()

        # Upsample block 1: H/32 -> H/16
        self.upsample_layer_0 = nn.ConvTranspose2d(1280, 1280, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(80, 1280, 1)  # Match features[3] channels
        self.decode_layer_0 = nn.Conv2d(1280, 768, 3, 1, 1)
        self.decode_layer_1 = nn.Conv2d(768, 768, 3, 1, 1)

        # Upsample block 2: H/16 -> H/8
        self.upsample_layer_1 = nn.ConvTranspose2d(768, 768, 2, 2)
        self.skip_link_layer_1 = nn.Conv2d(40, 768, 1)   # Match features[2] channels
        self.decode_layer_2 = nn.Conv2d(768, 512, 3, 1, 1)
        self.decode_layer_3 = nn.Conv2d(512, 512, 3, 1, 1)

        # Upsample block 3: H/8 -> H/4
        self.upsample_layer_2 = nn.ConvTranspose2d(512, 512, 2, 2)
        self.skip_link_layer_2 = nn.Conv2d(24, 512, 1)   # Match features[1] channels
        self.decode_layer_4 = nn.Conv2d(512, 512, 3, 1, 1)
        self.decode_layer_5 = nn.Conv2d(512, 256, 3, 1, 1)

    def forward(self, context, features):
        """
        Args:
            context: (B, 1280, H/32, W/32) from context module
            features: list of encoder features [l0, l2, l3, l4, l8]
                      features[1]=(B,24,H/4), features[2]=(B,40,H/8),
                      features[3]=(B,80,H/16)
        Returns:
            neck: (B, 256, H/4, W/4)
        """
        # Block 1: H/32 -> H/16
        d0 = self.upsample_layer_0(context)
        d0 = d0 + self.skip_link_layer_0(features[3])
        d1 = self.GeLU(self.decode_layer_0(d0))
        d2 = self.GeLU(self.decode_layer_1(d1))

        # Block 2: H/16 -> H/8
        d3 = self.upsample_layer_1(d2)
        d3 = d3 + self.skip_link_layer_1(features[2])
        d3 = self.GeLU(self.decode_layer_2(d3))
        d4 = self.GeLU(self.decode_layer_3(d3))

        # Block 3: H/8 -> H/4
        d5 = self.upsample_layer_2(d4)
        d5 = d5 + self.skip_link_layer_2(features[1])
        d5 = self.GeLU(self.decode_layer_4(d5))
        neck = self.GeLU(self.decode_layer_5(d5))

        return neck


neck_module = SceneNeck()

neck_output = neck_module(seg_ctx, features)
print(f"Neck output: {list(neck_output.shape)}")
print(f"Resolution: H/4 x W/4 = {320//4} x {640//4}")
print(f"\nThis 256-channel feature map is what all heads receive.")

## 3.4 — Heads: Task-Specific Output Layers

Heads are **lightweight** — they take the shared neck (256ch, H/4) and upsample to full resolution with a few conv layers and one more skip connection from `features[0]`.

- **SegmentationHead** → (B, 19, H, W) — 19-class logits
- **DepthHead** → (B, 1, H, W) — depth prediction

Since heads are small, they train fast. This is key for Module 3 where students will build their own!

In [ ]:
class SegmentationHead(nn.Module):
    """Semantic segmentation head.
    Upsamples neck from H/4 to H/1 and predicts per-pixel class labels.

    Args:
        num_classes: number of semantic classes (default=19 for BDD100K/Cityscapes)
    """

    def __init__(self, num_classes=19):
        super(SegmentationHead, self).__init__()
        self.GeLU = nn.GELU()

        # Upsample block 1: H/4 -> H/2
        self.upsample_layer_0 = nn.ConvTranspose2d(256, 256, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(32, 256, 1)  # Match features[0] channels
        self.decode_layer_0 = nn.Conv2d(256, 256, 3, 1, 1)
        self.decode_layer_1 = nn.Conv2d(256, 128, 3, 1, 1)

        # Upsample block 2: H/2 -> H
        self.upsample_layer_1 = nn.ConvTranspose2d(128, 128, 2, 2)
        self.decode_layer_2 = nn.Conv2d(128, 128, 3, 1, 1)
        self.decode_layer_3 = nn.Conv2d(128, 64, 3, 1, 1)

        # Output
        self.output_layer = nn.Conv2d(64, num_classes, 3, 1, 1)

    def forward(self, neck, features):
        """
        Args:
            neck: (B, 256, H/4, W/4) from SceneNeck
            features: encoder features list, uses features[0] for skip
        Returns:
            (B, num_classes, H, W) logits
        """
        # H/4 -> H/2
        d0 = self.upsample_layer_0(neck)
        d0 = d0 + self.skip_link_layer_0(features[0])
        d0 = self.GeLU(self.decode_layer_0(d0))
        d1 = self.GeLU(self.decode_layer_1(d0))

        # H/2 -> H
        d2 = self.upsample_layer_1(d1)
        d2 = self.GeLU(self.decode_layer_2(d2))
        d3 = self.GeLU(self.decode_layer_3(d2))

        output = self.output_layer(d3)
        return output


class DepthHead(nn.Module):
    """Monocular depth estimation head.
    Same structure as SegmentationHead but outputs 1 channel (depth map).
    """

    def __init__(self):
        super(DepthHead, self).__init__()
        self.GeLU = nn.GELU()

        # Upsample block 1: H/4 -> H/2
        self.upsample_layer_0 = nn.ConvTranspose2d(256, 256, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(32, 256, 1)
        self.decode_layer_0 = nn.Conv2d(256, 256, 3, 1, 1)
        self.decode_layer_1 = nn.Conv2d(256, 128, 3, 1, 1)

        # Upsample block 2: H/2 -> H
        self.upsample_layer_1 = nn.ConvTranspose2d(128, 128, 2, 2)
        self.decode_layer_2 = nn.Conv2d(128, 128, 3, 1, 1)
        self.decode_layer_3 = nn.Conv2d(128, 128, 3, 1, 1)

        # Output: single channel depth
        self.output_layer = nn.Conv2d(128, 1, 3, 1, 1)

    def forward(self, neck, features):
        """
        Args:
            neck: (B, 256, H/4, W/4) from SceneNeck
            features: encoder features list, uses features[0] for skip
        Returns:
            (B, 1, H, W) depth prediction
        """
        d0 = self.upsample_layer_0(neck)
        d0 = d0 + self.skip_link_layer_0(features[0])
        d0 = self.GeLU(self.decode_layer_0(d0))
        d1 = self.GeLU(self.decode_layer_1(d0))

        d2 = self.upsample_layer_1(d1)
        d2 = self.GeLU(self.decode_layer_2(d2))
        d3 = self.GeLU(self.decode_layer_3(d2))

        prediction = self.output_layer(d3)
        return prediction


seg_head = SegmentationHead(num_classes=19)
depth_head = DepthHead()

seg_out = seg_head(neck_output, features)
depth_out = depth_head(neck_output, features)

print(f"Segmentation output: {list(seg_out.shape)}  (19 class logits at full res)")
print(f"Depth output:        {list(depth_out.shape)}  (1 channel depth at full res)")

## 3.5 — The Full HydraNet

Now let's put it all together. The `HydraNet` class wires: Backbone → Context → Neck → Heads.

In [ ]:
class HydraNet(nn.Module):
    """Two-task HydraNet for Module 2: Segmentation + Depth.

    Architecture:
        Backbone (shared) -> Context (per-task) -> Neck (shared) -> Heads (per-task)

    Note: Both tasks share the same neck but have separate context modules,
    allowing each task to attend to different scene-level features.
    """

    def __init__(self, num_seg_classes=19):
        super(HydraNet, self).__init__()

        # Shared encoder
        self.backbone = Backbone(pretrained=True)

        # Task-specific context modules
        self.seg_context = SceneContext()
        self.depth_context = DepthContext()

        # Shared decoder neck
        self.neck = SceneNeck()

        # Task-specific heads
        self.seg_head = SegmentationHead(num_classes=num_seg_classes)
        self.depth_head = DepthHead()

    def forward(self, image):
        # Shared feature extraction
        features = self.backbone(image)
        deep_features = features[4]  # (B, 1280, H/32, W/32)

        # Segmentation branch
        seg_context = self.seg_context(deep_features)
        seg_neck = self.neck(seg_context, features)
        seg_output = self.seg_head(seg_neck, features)

        # Depth branch
        depth_context = self.depth_context(deep_features)
        depth_neck = self.neck(depth_context, features)
        depth_output = self.depth_head(depth_neck, features)

        return seg_output, depth_output

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_head_params(self):
        """Returns parameters for everything except the backbone
        (context, neck, heads) — useful for differential learning rates."""
        params = []
        for module in [self.seg_context, self.depth_context,
                       self.neck, self.seg_head, self.depth_head]:
            params.extend(module.parameters())
        return params


model = HydraNet(num_seg_classes=19).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.get_backbone_params())
head_params = sum(p.numel() for p in model.get_head_params())

print(f"Total parameters:    {total_params:,}")
print(f"Backbone parameters: {backbone_params:,} ({100*backbone_params/total_params:.1f}%)")
print(f"Context+Neck+Heads:  {head_params:,} ({100*head_params/total_params:.1f}%)")

# Test forward pass
dummy = torch.randn(2, 3, 320, 640).to(device)
seg_pred, depth_pred = model(dummy)
print(f"\nForward pass OK!")
print(f"Seg prediction:   {list(seg_pred.shape)}")
print(f"Depth prediction: {list(depth_pred.shape)}")

# 4 — Loss Functions

Multi-task learning requires combining multiple losses. We use:

- **Segmentation:** Cross-Entropy Loss (classification per pixel)
- **Depth:** Inverse Huber (berHu) Loss — L1 for small errors, L2 for large errors. More robust than pure L1 or L2.
- **Combined:** Weighted sum with learnable or fixed weights

The **Inverse Huber Loss** is defined as:
- If |error| ≤ c: loss = |error| (like L1)
- If |error| > c: loss = (error² + c²) / 2c (like L2)

where c = 0.2 × max(|error|) per batch.

In [ ]:
class InverseHuberLoss(nn.Module):
    """Inverse Huber (berHu) loss for depth estimation.

    Below threshold c: L1 loss (handles small errors well)
    Above threshold c: L2 loss (handles large errors well)

    c = 0.2 * max(|target - pred|) computed per batch.
    """

    def __init__(self):
        super(InverseHuberLoss, self).__init__()

    def forward(self, prediction, target):
        mask = target > 0  # Only compute loss where depth is valid
        prediction = prediction[mask]
        target = target[mask]

        if prediction.numel() == 0:
            return torch.tensor(0.0, device=prediction.device)

        diff = torch.abs(target - prediction)
        c = 0.2 * torch.max(diff).item()

        # berHu: L1 below c, L2 above c
        l1_mask = diff <= c
        l2_mask = diff > c

        loss = torch.zeros_like(diff)
        loss[l1_mask] = diff[l1_mask]
        if c > 0:
            loss[l2_mask] = (diff[l2_mask] ** 2 + c ** 2) / (2 * c)

        return loss.mean()


class MultiTaskLoss(nn.Module):
    """Combines multiple task losses with learnable or fixed weights.

    Args:
        task_names: list of task names
        learnable: if True, learns the loss weights (uncertainty weighting)
        initial_weights: dict of initial weights per task
    """

    def __init__(self, task_names, learnable=False, initial_weights=None):
        super(MultiTaskLoss, self).__init__()
        self.task_names = task_names
        self.learnable = learnable

        if initial_weights is None:
            initial_weights = {name: 1.0 for name in task_names}

        if learnable:
            # Learnable log-variance (Kendall et al., 2018)
            self.log_vars = nn.ParameterDict({
                name: nn.Parameter(torch.tensor(0.0))
                for name in task_names
            })
        else:
            self.weights = initial_weights

    def forward(self, losses):
        """
        Args:
            losses: dict of {task_name: loss_value}
        Returns:
            total_loss, loss_dict (for logging)
        """
        total = torch.tensor(0.0, device=list(losses.values())[0].device)
        loss_dict = {}

        for name in self.task_names:
            if name not in losses:
                continue

            if self.learnable:
                precision = torch.exp(-self.log_vars[name])
                weighted = precision * losses[name] + self.log_vars[name]
            else:
                weighted = self.weights[name] * losses[name]

            total = total + weighted
            loss_dict[name] = losses[name].item()

        loss_dict['total'] = total.item()
        return total, loss_dict


# Individual task losses
seg_criterion = nn.CrossEntropyLoss(ignore_index=255)
depth_criterion = InverseHuberLoss()

# Multi-task loss combiner
mtl_loss = MultiTaskLoss(
    task_names=['seg', 'depth'],
    learnable=False,
    initial_weights={'seg': 1.0, 'depth': 1.0}
)

print("Loss functions ready!")

# 5 — Training

We use **differential learning rates**: a lower LR for the pretrained backbone, and a higher LR for the new layers (context, neck, heads). This prevents destroying the pretrained features while allowing the new layers to learn quickly.

In [ ]:
# Optimizer with differential learning rates
optimizer = torch.optim.SGD([
    {'params': model.get_backbone_params(), 'lr': 1e-4},    # Low LR for pretrained backbone
    {'params': model.get_head_params(), 'lr': 1e-2},        # High LR for new layers
], momentum=0.9, weight_decay=1e-5)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[15, 25], gamma=0.1
)

NUM_EPOCHS = 30
print(f"Training for {NUM_EPOCHS} epochs")
print(f"Backbone LR: {optimizer.param_groups[0]['lr']}")
print(f"Heads LR:    {optimizer.param_groups[1]['lr']}")

In [ ]:
def compute_miou(predictions, targets, num_classes, ignore_index=255):
    """Compute Mean Intersection over Union for segmentation.

    Args:
        predictions: (B, num_classes, H, W) logits
        targets: (B, H, W) integer class labels
        num_classes: number of classes
        ignore_index: label index to ignore

    Returns:
        miou: float
    """
    preds = predictions.argmax(dim=1)  # (B, H, W)
    ious = []

    for cls in range(num_classes):
        pred_mask = (preds == cls)
        target_mask = (targets == cls)
        valid = (targets != ignore_index)

        pred_mask = pred_mask & valid
        target_mask = target_mask & valid

        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()

        if union > 0:
            ious.append((intersection / union).item())

    return np.mean(ious) if ious else 0.0


def compute_rmse(prediction, target):
    """Compute Root Mean Squared Error for depth estimation.

    Args:
        prediction: (B, 1, H, W) predicted depth
        target: (B, 1, H, W) ground truth depth

    Returns:
        rmse: float
    """
    mask = target > 0  # Only evaluate where depth is valid
    if mask.sum() == 0:
        return 0.0

    diff = prediction[mask] - target[mask]
    return torch.sqrt((diff ** 2).mean()).item()


def train_one_epoch(model, loader, optimizer, seg_criterion, depth_criterion, mtl_loss, device):
    model.train()
    epoch_losses = {'seg': 0, 'depth': 0, 'total': 0}
    n_batches = 0

    for batch in tqdm(loader, desc='Training', leave=False):
        images = batch['image'].to(device)
        seg_gt = batch['seg'].to(device)
        depth_gt = batch['depth'].to(device)

        # Forward pass
        seg_pred, depth_pred = model(images)

        # Compute individual losses
        loss_seg = seg_criterion(seg_pred, seg_gt)
        loss_depth = depth_criterion(depth_pred, depth_gt)

        # Combine losses
        total_loss, loss_dict = mtl_loss({'seg': loss_seg, 'depth': loss_depth})

        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        for k in epoch_losses:
            epoch_losses[k] += loss_dict[k]
        n_batches += 1

    return {k: v / n_batches for k, v in epoch_losses.items()}


@torch.no_grad()
def validate(model, loader, seg_criterion, depth_criterion, mtl_loss, device, num_classes=19):
    model.eval()
    epoch_losses = {'seg': 0, 'depth': 0, 'total': 0}
    total_miou = 0
    total_rmse = 0
    n_batches = 0

    for batch in tqdm(loader, desc='Validation', leave=False):
        images = batch['image'].to(device)
        seg_gt = batch['seg'].to(device)
        depth_gt = batch['depth'].to(device)

        seg_pred, depth_pred = model(images)

        loss_seg = seg_criterion(seg_pred, seg_gt)
        loss_depth = depth_criterion(depth_pred, depth_gt)
        _, loss_dict = mtl_loss({'seg': loss_seg, 'depth': loss_depth})

        for k in epoch_losses:
            epoch_losses[k] += loss_dict[k]

        # Metrics
        total_miou += compute_miou(seg_pred, seg_gt, num_classes)
        total_rmse += compute_rmse(depth_pred, depth_gt)
        n_batches += 1

    avg_losses = {k: v / n_batches for k, v in epoch_losses.items()}
    avg_losses['miou'] = total_miou / n_batches
    avg_losses['rmse'] = total_rmse / n_batches
    return avg_losses

print("Training functions defined!")

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_rmse': []}
best_miou = 0.0

for epoch in range(NUM_EPOCHS):
    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, seg_criterion, depth_criterion, mtl_loss, device
    )

    # Validate
    val_metrics = validate(
        model, val_loader, seg_criterion, depth_criterion, mtl_loss, device
    )

    scheduler.step()

    # Log
    history['train_loss'].append(train_metrics['total'])
    history['val_loss'].append(val_metrics['total'])
    history['val_miou'].append(val_metrics['miou'])
    history['val_rmse'].append(val_metrics['rmse'])

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_metrics['total']:.4f} | "
          f"Val Loss: {val_metrics['total']:.4f} | "
          f"mIoU: {val_metrics['miou']:.4f} | "
          f"RMSE: {val_metrics['rmse']:.4f}")

    # Save best model
    if val_metrics['miou'] > best_miou:
        best_miou = val_metrics['miou']
        torch.save(model.state_dict(), 'hydranet_best.pth')
        print(f"  -> Saved best model (mIoU: {best_miou:.4f})")

print(f"\nTraining complete! Best mIoU: {best_miou:.4f}")

# 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# mIoU
axes[1].plot(history['val_miou'], color='green')
axes[1].set_title('Validation mIoU (Segmentation)')
axes[1].set_xlabel('Epoch')

# RMSE
axes[2].plot(history['val_rmse'], color='orange')
axes[2].set_title('Validation RMSE (Depth)')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

# 7 — Inference & Visualization

Let's load the best model and visualize predictions on validation images.

In [ ]:
# Load best model
model.load_state_dict(torch.load('hydranet_best.pth', map_location=device))
model.eval()

# Denormalize for visualization
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(img_tensor):
    return (img_tensor.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

# Predict on a few validation images
val_batch = next(iter(val_loader))
images = val_batch['image'].to(device)

with torch.no_grad():
    seg_pred, depth_pred = model(images)

seg_pred = seg_pred.argmax(dim=1).cpu().numpy()
depth_pred = depth_pred.squeeze(1).cpu().numpy()

# Visualize
n_show = min(4, images.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))

for i in range(n_show):
    # RGB
    axes[i, 0].imshow(denormalize(images[i]))
    axes[i, 0].set_title('Input Image')
    axes[i, 0].axis('off')

    # GT Seg
    axes[i, 1].imshow(colorize_mask(val_batch['seg'][i].numpy()))
    axes[i, 1].set_title('GT Segmentation')
    axes[i, 1].axis('off')

    # Predicted Seg
    axes[i, 2].imshow(colorize_mask(seg_pred[i]))
    axes[i, 2].set_title('Predicted Segmentation')
    axes[i, 2].axis('off')

    # Predicted Depth
    axes[i, 3].imshow(depth_pred[i], cmap='magma')
    axes[i, 3].set_title('Predicted Depth')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

# 8 — FPS Measurement

Let's measure how fast our HydraNet runs — an important metric for autonomous driving!

In [ ]:
import time

model.eval()
dummy = torch.randn(1, 3, 320, 640).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)

# Benchmark
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()
n_runs = 100
for _ in range(n_runs):
    with torch.no_grad():
        _ = model(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start
fps = n_runs / elapsed
print(f"Average inference time: {1000 * elapsed / n_runs:.1f} ms")
print(f"FPS: {fps:.1f}")
print(f"\nNote: Both segmentation AND depth are computed in a single forward pass!")

# 9 — Save Pre-computed Features for Module 3

We save the backbone features for the validation set so students in Module 3 can train new heads **without running the backbone** — making training fast enough for a lab session.

In [ ]:
import os

os.makedirs('precomputed', exist_ok=True)

# Save model weights
torch.save(model.state_dict(), 'precomputed/hydranet_module2.pth')

# Pre-compute and save backbone + neck features for the training set
model.eval()
all_necks = []
all_features_0 = []
all_filenames = []

print("Pre-computing features for Module 3...")
with torch.no_grad():
    for batch in tqdm(train_loader, desc='Pre-computing'):
        images = batch['image'].to(device)

        # Run through backbone and neck
        features = model.backbone(images)
        deep_features = features[4]
        context = model.seg_context(deep_features)
        neck = model.neck(context, features)

        all_necks.append(neck.cpu())
        all_features_0.append(features[0].cpu())
        all_filenames.extend(batch['filename'])

# Save
torch.save({
    'neck': torch.cat(all_necks, dim=0),
    'features_0': torch.cat(all_features_0, dim=0),
    'filenames': all_filenames,
}, 'precomputed/train_features.pt')

print(f"Saved {len(all_filenames)} pre-computed feature sets to precomputed/")
print(f"Neck shape: {torch.cat(all_necks, dim=0).shape}")
print(f"Features[0] shape: {torch.cat(all_features_0, dim=0).shape}")
print(f"\nStudents can now train new heads in minutes!")

# Next Steps

In **Module 3**, you'll learn how to add new heads to this pre-trained HydraNet:
- **Lane Detection** — ego-lane segmentation
- **2D Object Detection** — anchor-free CenterNet-style detection
- **Trajectory Prediction** — steering angle prediction (like Autoware Vision Pilot)

The backbone is frozen, so training new heads takes only **5-15 minutes** on Colab!